In [ ]:
print("="*70)
print("ArbItro Test Pipeline - Google Colab")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f" GPU: {gpu_info[0].strip()}")
    print(f" VRAM: {gpu_info[1].strip()}")
    print("\n System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n No GPU found")

print("\n" + "="*70)

In [ ]:
import os, sys

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR      = '/content/drive/MyDrive/ArbItro'
    CODE_DIR      = os.path.join(BASE_DIR, 'ArbItro_code')
    DATASET_DIR   = os.path.join(BASE_DIR, 'ArbItro_dataset', 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False
    BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
    CODE_DIR      = os.path.join(BASE_DIR, 'model', 'src', 'pipeline1')
    DATASET_DIR   = os.path.join(BASE_DIR, 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, CODE_DIR)

print(f"\n  USE_COLAB   : {USE_COLAB}")
print(f"  Dataset dir : {DATASET_DIR}")
print(f"  Models dir  : {os.path.join(TRAINING_ROOT, 'models')}")

required_paths = [
    os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    os.path.join(DATASET_DIR, 'test'),
]

print("\nVerifying test paths:")
all_ok = True
for path in required_paths:
    exists = os.path.exists(path)
    status = " Found" if exists else " Not Found"
    if not exists:
        all_ok = False
    print(f"  {status}: {path}")

In [ ]:
# Install required packages
!pip install opencv-python-headless

In [ ]:
if USE_COLAB:
    !mkdir -p /content/arbitro_model
    %cd /content/arbitro_model
    !cp {CODE_DIR}/data_loader.py .
    !cp {CODE_DIR}/model.py .
    sys.path.insert(0, '/content/arbitro_model')
    print("\nFiles copied to working directory:")
    !ls -lh

In [ ]:
try:
    from data_loader import ArbItroDataGenerator
    from model import build_arbitro_model_speed_aware, compile_arbitro_model
    print("  Imports successful!")
except Exception as e:
    print(f"  Import error: {e}")
    raise

In [ ]:
TEST_CONFIG = {
    'json_path'  : os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    'video_dir'  : os.path.join(DATASET_DIR, 'test'),
    'model_path' : os.path.join(TRAINING_ROOT, 'models', 'final_model_fine_tuned.keras'),
    'batch_size' : 1,
    'dim'        : (224, 398),
    'n_frames'   : 16,
    'shuffle'    : False,
}

ACTION_MAP = {
    "": 0, "Dont know": 0, "Challenge": 1, "Tackling": 2,
    "Standing tackling": 3, "High leg": 4, "Holding": 5,
    "Pushing": 6, "Elbowing": 7, "Dive": 8
}
INV_ACTION_MAP   = {v: k for k, v in ACTION_MAP.items() if k != ""}
INV_SEVERITY_MAP = {0: "No Card", 1: "Yellow", 2: "Red"}
INV_OFFENCE_MAP  = {0: "No Offence", 1: "Offence"}


print(f"  Model path: {TEST_CONFIG['model_path']}")
if not os.path.exists(TEST_CONFIG['model_path']):
    print(" Model not found")
else:
    print(" Model found")

In [ ]:
import traceback

try:
    test_gen = ArbItroDataGenerator(
        json_path=TEST_CONFIG['json_path'],
        base_video_path=TEST_CONFIG['video_dir'],
        batch_size=TEST_CONFIG['batch_size'],
        dim=TEST_CONFIG['dim'],
        n_frames=TEST_CONFIG['n_frames'],
        shuffle=False,
        use_auxiliary_features=True,
        augment=False
    )
    print(f" Test samples (originals): {test_gen.n_samples}")
    print(f" Test batches: {len(test_gen)}")

except Exception as e:
    print(f" Generator error: {e}")
    traceback.print_exc()
    raise

try:
    model = tf.keras.models.load_model(TEST_CONFIG['model_path'], compile=False)
    print(f"\n  Model loaded: {TEST_CONFIG['model_path']}")
    print(f"  Output heads    : {model.output_names}")
except Exception as e:
    print(f"  Error during loading model: {e}")
    traceback.print_exc()
    raise